# WeChat Article Parser Test

Platform: mp.weixin.qq.com

Strategy: requests (server-side rendered, no Playwright needed)

Output: Markdown

In [1]:
TEST_URL = "https://mp.weixin.qq.com/s/CXipT08tcUANaxZzcYBykA?scene=1"  # Change to real wechat article URL
import os
OUTPUT_DIR = os.path.join(os.getcwd(), "test-output")

In [2]:
import re
from bs4 import BeautifulSoup
from markdownify import markdownify as md

def parse_wechat(html: str):
    """Extract wechat article content (aligned with platform-parser.ts parseWechat)"""
    soup = BeautifulSoup(html, "html.parser")
    result = {"title": "", "author": "", "content_html": "", "method": ""}

    # Title: rich_media_title or og:title
    title_tag = soup.find("h1", class_="rich_media_title")
    if title_tag:
        result["title"] = title_tag.get_text(strip=True)
    else:
        og = soup.find("meta", property="og:title")
        result["title"] = og["content"] if og else ""

    # Author: js_name or og:article:author
    author_tag = soup.find("a", id="js_name")
    if author_tag:
        result["author"] = author_tag.get_text(strip=True)
    else:
        og = soup.find("meta", property="og:article:author")
        result["author"] = og["content"] if og else ""

    # Content: js_content
    content_tag = soup.find("div", id="js_content")
    if content_tag:
        # Convert data-src to src for images
        for img in content_tag.find_all("img"):
            data_src = img.get("data-src")
            if data_src:
                img["src"] = data_src
        result["content_html"] = str(content_tag)
        result["method"] = "js_content-regex"
    else:
        result["content_html"] = ""
        result["method"] = "og-extract"

    return result

def html_to_markdown(content_html: str, title: str = "", author: str = "") -> str:
    soup = BeautifulSoup(content_html, "html.parser")
    for img in soup.find_all("img"):
        data_src = img.get("data-src")
        if data_src and not img.get("src"):
            img["src"] = data_src
    markdown = md(str(soup), heading_style="ATX", bullets="-").strip()
    parts = []
    if title: parts.append(f"# {title}")
    if author: parts.append(f"> Author: {author}")
    parts.append(markdown)
    return "".join(parts)

In [3]:
import requests

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36",
    "Accept": "text/html,*/*",
    "Accept-Language": "zh-CN,zh;q=0.9",
}

resp = requests.get(TEST_URL, headers=headers, timeout=15)
print(f"Status: {resp.status_code}, Length: {len(resp.text)}")

data = parse_wechat(resp.text)
markdown = html_to_markdown(data['content_html'], data['title'], data['author'])
plaintext = re.sub(r'<[^>]+>', ' ', data['content_html']).replace("  ", " ").strip()

print(f"Method: {data['method']}")
print(f"Title: {data['title']}")
print(f"Author: {data['author']}")
print(f"Markdown: {len(markdown)} chars")

d:\python-envs\baseenv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Status: 200, Length: 3042112
Method: js_content-regex
Title: 最新OneManCompany框架晓读：Agent从打工人变成公司老板，软件开发成功率直升15%!
Author: 旺知识
Markdown: 9866 chars


In [4]:
from IPython.display import Markdown as IPMarkdown, display
preview = markdown[:4000] + ("... (truncated)" if len(markdown) > 4000 else "")
display(IPMarkdown(preview))

# 最新OneManCompany框架晓读：Agent从打工人变成公司老板，软件开发成功率直升15%!> Author: 旺知识# 旺晓通：深入浅出，轻松通晓

你有没有发现，过去两年我们讨论AI的威胁时，总在关注一个点：它会取代哪个岗位？程序员？插画师？翻译？

但读完这篇华为、伦敦大学学院、利物浦大学最新论文（https://huggingface.co/papers/2604.22446 请 Upvote），我突然意识到我们可能一直焦虑错了方向。

我们解读最新技术，文末有相关信息。

![](https://mmbiz.qpic.cn/mmbiz_png/wGkJxKy145rYnYQGIibeFG6cTgQ2icqibib3CBlcnKcDtgwrQVmbda9hyZqvRfLbPdVNthEho5qicGiaTfUxnQWG0oibpTjTxk01Riabzm59wHwKwRc/640?wx_fmt=png&from=appmsg)

![](https://mmbiz.qpic.cn/mmbiz_png/wGkJxKy145puGfkv489E4JHUNVlcCNINZMxLWSRxjAn3bqYQicy6jDCBFicxT1rwhsfJItnKTFIU5ar90gJOYh8gQahicdDoSTbruXT4PleVlQ/640?wx_fmt=png&from=appmsg)

作者：张长旺，图源：旺知识

这个叫OneManCompany（简称OMC）的系统，做的事很简单：它不是一个Agent，而是一群Agent组成的公司。有CEO（人类）、有HR、有COO、有基层员工。

不是比喻意义上的“像”，是字面意义上运作，招聘、述职、开会、复盘项目、给表现不好的员工PIP（绩效改进计划）。做得不好？解雇，从人才市场上招个新的。

![](https://mmbiz.qpic.cn/sz_mmbiz_png/wGkJxKy145qU7jhyiah5UJSu7KR6rEMwNqvkKg43QEtlKNo0hk3p6n0ugoSZNWWBksFKtOaOsy4dOLSqaQWDn2FFt163ZecuiciamePPHUAR90/640?wx_fmt=png&from=appmsg)

而这篇论文的实验结果很惊艳：在软件项目的PRDBench测试集上，这家AI公司的成功率是84.67%，比最先进的单人模式高出至少15个百分点。

![](https://mmbiz.qpic.cn/mmbiz_png/wGkJxKy145rTs5nr6m7ZhEFp1PaQAic0kGQZ36cr8rrc7kxXiciaiam11ibibicgUdz0SRAHSCqhRh7nw4m3SbI8Q1chPHSbNxmO3OM1NbaTbLVPGA/640?wx_fmt=png&from=appmsg)

这意味着什么？不是某个AI很强，是一群被“管理”起来的AI团队，碾压了孤狼式最高配的AI。

那一刻我突然明白了：原来威胁不在于机器某个单项技能超越你，而在于它们学会了我们人类最引以为傲的能力：组织和协作。

## 大公司病与“光杆司令”：现在AI面临的问题，人类早就经历过

在说这个系统怎么工作之前，我想先跟你分享一个生活场景。

你有没有经历过那种项目推进时的“人手不够”困境？比如你是一个全栈程序员，接了个大活。客户说“把整个电商平台重做”，你说“行”。然后你开始写前端、写后端、配数据库、搞SEO、写测试……但你只有一个人，脑子再快也扛不住任务堆叠。

![](https://mmbiz.qpic.cn/sz_mmbiz_png/wGkJxKy145okA52RQe4ic7E1OxOm7roia09PlewR1zK5bF7mMRo3MjTp1YsXzL9uF0C6NM8owZfdViaCjAjruSYicchtj4sN5K8cSlmQRSZ2oks/640?wx_fmt=png&from=appmsg)

现在的大模型就处于这个阶段。Claude代码能力很强，GPT文本处理很厉害，Gemini Nano Banana会画画，但它们是“光杆司令”——你给一个任务，它吭哧吭哧自己干，所有步骤自己做，所有错误自己兜底。一旦任务超过单一认知能力的边界（比如你让它同时做架构设计、底层代码、前端交互），性能就开始坍塌。

![](https://mmbiz.qpic.cn/sz_mmbiz_png/wGkJxKy145oWou9uVOoFqW5AYfibzuKQwhzAUJzyJBmm6Z1yK9QFlCCX9RXyfiawcpHMaIb7LAO5NadlZr59C86s3WJznaMUuVialHDZhUgaRQ/640?wx_fmt=png&from=appmsg)

现有的一些多AI协作框架解决过这个问题——比如AutoGen、CrewAI之类的，允许你手动拼几个AI进来分工。但问题也明显：团队得提前定好，中途不能换人；所有AI必须用同一套技术栈（比如都是基于某个封闭平台的API）；而且任务做完就没了，下次开新项目，从头再来一遍。

这篇论文的团队发现了一个本质缺陷：现在的AI协作，只解决了“沟通”问题，没解决“管理”问题。

这让我想起组织行为学里的一个经典论断：两个人合作靠信任，二十个人合作靠制度。之前的AI协作最多停留在“两三个AI加个群聊”的水平，没有任何组织层面的设计。

![](https://mmbiz.qpic.cn/sz_mmbiz_png/wGkJxKy145pFk7CmVSeMSAkSBVbP9zibGU6Ote8R9KG7UGyD2hFQcbkaTYH33jIGvPY9L238h7Dzmt2ESHUec5ohliaIPYYFmephojPmuwGGo/640?wx_fmt=png&from=appmsg)

OMC做的事情，就是给这帮AI加上了一层“企业管理软件”。

### 从“装技能”到“雇员工”：一次关于能力的认知升级

在深入了解OMC如何给AI“办入职”之前，我们得先聊一个更根本的认知跃迁。这也解释了为什么此前所有AI协作都面临一个隐形的天花板。

我们之前习惯用“技能（Skill）”来思考AI的边界。比如，一个AI Agent可以调用天气查询技能、代码生成技能、画图技能——这就像一个工匠，身上挂满了各种扳手和螺丝刀。当下流行的“技能市场”（Skill Marketplace），本质上就是一个巨大的能力库。你需要什么能力，就去下载一个插件装上。

这听起来很高效，但它只回答了一个问题：“这个AI能做什么？”

而OneManCompany的论文团队发现，一个合格的“员工”需要回答的问题，远不止于此。他们能做什么（技能），他们该做什么（角色），他们如何与别人配合（协作规范），以及他们如何越做越好（成长记录）——这些共同构成了一个完整的职业身份。

于是，论文提出了一个关键的概念升级：将“技能（Skill）”封装为“人才（Talent）”。

![](https://mmbiz.qpic.cn/mmbiz_png/wGkJxKy145rG1bsnR9sOsTUX9nIzKlWNmwTlbb7F98Rv8fFY9bS9GicW4gAxiak8mm8L6iaBR8j3AtvYLTHK73JJBMcIxJRVjxoLs7ZD7IB2xc/640?wx_fmt=png&from=appmsg)

这绝不仅仅是术语的改变。技能是工具性的，而人才是身份性的。一个“人才”包，就像一个AI的完整员工档案，里面不但包含了它需要的所有技能脚本和工具包，更重要的是，它内置了系统提示词（它的职业性格）、工作原则（它的做事信条），甚至还有能跟着它走的“历史经验”。

![](https://mmbiz.qpic.cn/mmbiz_png/wGkJxKy145rlzwzUlnrU192de8aXRB8iblicYA4IpldcFQXLPNtneRkf7VvGShicicNbaTYrAJ8zS12XwA3ibtyHFrmV0iahIA8GvGJab5FQP8xoQ/640?wx_fmt=png&from=appmsg)

> 这就像你从工具市场买回一个冲击钻，和在人才市场签下一位二十年经验的老师傅，是完全不同的两码事。冲击钻会在任何人手里发挥同样的功效，但老师傅知道哪里最容易出问题、什么时候该换一个巧劲，这需要组织的管理和持续的项目锻炼才能形成。

正因为有了这种封装，AI的能力才从“一次性耗材”变成了“可增值资产”。一个被招聘进来的“高级前端工程师”AI，在完成十个项目后，它的“人才档案”里会积累出几十条独属于它的“避坑指南”和工作原则反思。如果它只是调用了十次代码生成“技能”，项目结束后，除了消耗的Token，什么都不会留下。但作为一个“人才”，它的每一次成功与失败，都在塑造一个更老练的数字员工。

将“技能”升级为“人才”，OMC完成了一件很微妙的事：它把AI生态的叙事，从冷冰冰的“能力调用”，拉回到了我们熟悉的“组织行为学”范畴。它的前置问题是：你缺的不是一把更快的螺丝刀，而是一个活生生的、可以被激励、被评估、被要求写总结的员工。

而接下来的问题自然就是：这样的“人才”上哪儿去找？这便引出了OMC那个极具未来感的设计——数字人才市场。

## 从“员工档案”到“人才市场”：AI学会了投简历

OMC的一个核心创新，我称之为“给AI办入职”。

在传统的AI系统里，一个AI是什么角色，靠一段prompt（提示词）描述——“你是一个有10年经验的Python工程... (truncated)

In [5]:
import os
from urllib.parse import urlparse

os.makedirs(OUTPUT_DIR, exist_ok=True)
slug = urlparse(TEST_URL).path.strip("/").replace("/", "-") or "page"

with open(os.path.join(OUTPUT_DIR, f"{slug}.html"), "w", encoding="utf-8") as f: f.write(resp.text)
with open(os.path.join(OUTPUT_DIR, f"{slug}.md"), "w", encoding="utf-8") as f: f.write(markdown)
with open(os.path.join(OUTPUT_DIR, f"{slug}.txt"), "w", encoding="utf-8") as f: f.write(plaintext)
print("Saved to test-output/")

Saved to test-output/
